# Detecção de Objetos em Tempo Real (YOLOv11)

Este notebook detecta qualquer um dos 80 objetos padrão do COCO (carros, cachorros, cadeiras, etc.) usando a webcam.

### Step 1: Configuração
```bash
pip install ultralytics opencv-python torch torchvision ipywidgets
```

In [ ]:
import cv2
import torch
import os
import time
from ultralytics import YOLO
import ipywidgets as widgets
from IPython.display import display

# Previne erro de biblioteca duplicada no Windows
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando: {device}")

try:
    # Carrega modelo Nano (rápido)
    model = YOLO("yolo11n.pt")
    model.to(device)
    print("Modelo pronto para detectar qualquer objeto.")
except Exception as e:
    print(f"Erro ao carregar: {e}")

### Step 2: Iniciar WebCam (Exibição Estabilizada)

- Não abre janela externa (exibe aqui mesmo).
- Para parar, clique no botão **Stop** do notebook.

In [ ]:
image_widget = widgets.Image(format='jpeg', width=640, height=480)
display(image_widget)

cap = cv2.VideoCapture(0)
prev_time = 0

if not cap.isOpened():
    print("Erro: Câmera não encontrada.")
else:
    print("Detectando... Clique no Stop para encerrar.")
    
    try:
        while True:
            success, frame = cap.read()
            if not success: break

            # Chamada sem filtro 'classes', detecta tudo
            results = model(frame, verbose=False, conf=0.5, device=device)
            
            # Plotar resultados (bounding boxes e nomes)
            annotated_frame = results[0].plot()

            # FPS
            curr_time = time.time()
            fps = 1 / (curr_time - prev_time) if (curr_time - prev_time) > 0 else 0
            prev_time = curr_time
            
            cv2.putText(annotated_frame, f"FPS: {int(fps)}", (20, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            # Atualizar imagem no notebook
            _, buffer = cv2.imencode('.jpg', annotated_frame)
            image_widget.value = buffer.tobytes()
            
    except KeyboardInterrupt: pass
    finally:
        cap.release()
        print("Câmera liberada.")